In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm


In [ ]:
VALIDATION_LOOP_DIR = Path(".").resolve()
REPO_DIR = VALIDATION_LOOP_DIR.parent

sys.path.insert(0, str(REPO_DIR / "training_loop"))
sys.path.insert(0, str(REPO_DIR))

DATASET_DIR = REPO_DIR.parent / "dataset"
TRAIN_DIR = DATASET_DIR / "Dataset_train"
VAL_DIR = DATASET_DIR / "Dataset_validation"
TRAIN_CSV = REPO_DIR / "tags" / "train.csv"
VAL_CSV = REPO_DIR / "tags" / "validation.csv"
OUTPUT_DIR = REPO_DIR / "output" / "history_3d_simple_cnn_multilabel_1"

label_columns = [
    "epidural hemorrhage",
    "subarachnoid hemorrhage",
    "subdural hemorrhage",
    "intracerebral hemorrhage",
]

target_size = 128
epoch = 18
batch_size_train = 8
batch_size_val = 8
num_workers = 4


In [ ]:
from dataset import CTVolumeDataset


class ConvBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout_p=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.InstanceNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout3d(p=dropout_p) if dropout_p > 0 else nn.Identity(),
        )

    def forward(self, x):
        return self.block(x)


class Simple3DClassifier(nn.Module):
    def __init__(self, in_channels=1, num_classes=1):
        super().__init__()

        self.encoder = nn.Sequential(
            ConvBlock3D(in_channels, 16, stride=2, dropout_p=0.1),
            ConvBlock3D(16, 32, stride=2, dropout_p=0.1),
            ConvBlock3D(32, 64, stride=2, dropout_p=0.1),
            ConvBlock3D(64, 128, stride=2, dropout_p=0.1),
            ConvBlock3D(128, 256, stride=2, dropout_p=0.1),
        )

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.head(x)
        return x


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_dataset = CTVolumeDataset(
    table_path=TRAIN_CSV,
    images_dir=TRAIN_DIR,
    label_columns=label_columns,
    target_size=target_size,
)

val_dataset = CTVolumeDataset(
    table_path=VAL_CSV,
    images_dir=VAL_DIR,
    label_columns=label_columns,
    target_size=target_size,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size_train,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=(device.type == "cuda"),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size_val,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=(device.type == "cuda"),
)

model = Simple3DClassifier(in_channels=1, num_classes=len(label_columns)).to(device)
checkpoint = torch.load(OUTPUT_DIR / f"checkpoint_epoch_{epoch}.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()


In [ ]:
def collect_split_predictions(model, loader, device, split_name):
    scores = []

    with torch.no_grad():
        for volumes, _ in tqdm(loader, desc=f"Predict {split_name}", leave=False):
            volumes = volumes.to(device, non_blocking=True)
            logits = model(volumes)
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            scores.append(probs)

    return np.concatenate(scores, axis=0)


train_y_score = collect_split_predictions(
    model=model,
    loader=train_loader,
    device=device,
    split_name="train",
)

val_y_score = collect_split_predictions(
    model=model,
    loader=val_loader,
    device=device,
    split_name="validation",
)


In [ ]:
prob_columns = [f"prob_{col}" for col in label_columns]

train_pred_df = pd.DataFrame(train_y_score, columns=prob_columns)
train_pred_df.insert(0, "study_uid", train_dataset.samples_df["study_uid"].reset_index(drop=True))

val_pred_df = pd.DataFrame(val_y_score, columns=prob_columns)
val_pred_df.insert(0, "study_uid", val_dataset.samples_df["study_uid"].reset_index(drop=True))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_pred_df.to_csv(OUTPUT_DIR / "train_prediction.csv", index=False)
val_pred_df.to_csv(OUTPUT_DIR / "validation_predictions.scv", index=False)

print("Saved:")
print(OUTPUT_DIR / "train_prediction.csv")
print(OUTPUT_DIR / "validation_predictions.scv")
